# Agent 1: User Profile Collector

### Agent Responsibility
> *"Collects user profile from free-text input or direct prompts, extracts age/weight/height/gender/goal via regex, then asks for anything still missing"*

In [1]:
import re

## User Profile Agent

In [2]:
class UserProfileAgent:

    def __init__(self):
        self.profile = {
            "age":            None,
            "gender":         None,
            "weight_kg":      None,
            "height_cm":      None,
            "activity_level": None,
            "goal":           None
        }

    def extract_from_text(self, text):
        text = text.lower()

        def find(pattern):
            match = re.search(pattern, text)
            return float(match.group(1)) if match else None

        age    = find(r'(\d+)\s*(year|years|yo|y\.o)')
        weight = find(r'(\d+(?:\.\d+)?)\s*(kg|kgs|kilogram)')
        height = find(r'(\d+(?:\.\d+)?)\s*(cm|centimeter)')

        if age:    self.profile["age"]       = int(age)
        if weight: self.profile["weight_kg"] = float(weight)
        if height: self.profile["height_cm"] = float(height)

        if "male" in text or "man" in text or " guy" in text:
            self.profile["gender"] = "male"
        elif "female" in text or "woman" in text or "girl" in text:
            self.profile["gender"] = "female"

        if any(w in text for w in ["lose", "fat", "cut", "slim", "shred"]):
            self.profile["goal"] = "weight_loss"
        elif any(w in text for w in ["gain", "muscle", "bulk", "mass"]):
            self.profile["goal"] = "muscle_gain"
        elif "maintain" in text or "stay" in text:
            self.profile["goal"] = "maintenance"

        if any(w in text for w in ["sedentary", "no exercise", "inactive", "desk"]):
            self.profile["activity_level"] = "sedentary"
        elif any(w in text for w in ["light", "walk", "once a week"]):
            self.profile["activity_level"] = "light"
        elif any(w in text for w in ["moderate", "3x", "sometimes"]):
            self.profile["activity_level"] = "moderate"
        elif any(w in text for w in ["active", "daily", "intense", "athlete", "5x"]):
            self.profile["activity_level"] = "active"

    def safe_input(self, prompt, cast=str):
        while True:
            val = input(prompt)
            try:
                return cast(val)
            except:
                print("Invalid input, try again.")

    def ask_missing(self):
        if self.profile["age"] is None:
            self.profile["age"] = self.safe_input("Enter age: ", int)
        if self.profile["gender"] is None:
            self.profile["gender"] = self.safe_input("Gender (male/female): ")
        if self.profile["weight_kg"] is None:
            self.profile["weight_kg"] = self.safe_input("Weight (kg): ", float)
        if self.profile["height_cm"] is None:
            self.profile["height_cm"] = self.safe_input("Height (cm): ", float)
        if self.profile["activity_level"] is None:
            print("Activity levels: sedentary / light / moderate / active")
            self.profile["activity_level"] = self.safe_input("Activity level: ")
        if self.profile["goal"] is None:
            print("Goals: weight_loss / muscle_gain / maintenance")
            self.profile["goal"] = self.safe_input("Goal: ")

    def validate(self):
        errors = []
        if not (10 <= self.profile["age"] <= 100):
            errors.append("Invalid age")
        if not (30 <= self.profile["weight_kg"] <= 300):
            errors.append("Invalid weight")
        if not (100 <= self.profile["height_cm"] <= 250):
            errors.append("Invalid height")
        if self.profile["gender"] not in ("male", "female"):
            errors.append("Invalid gender")
        if errors:
            raise ValueError(", ".join(errors))

    def run(self):
        print("Describe yourself (optional, press Enter to skip):")
        text = input("> ")
        if text.strip():
            self.extract_from_text(text)
            print("Extracted:", {k: v for k, v in self.profile.items() if v is not None})
        self.ask_missing()
        self.validate()
        return self.profile

## Testing — Regex Extraction
Test what gets extracted from free-text without prompting the user

In [3]:
test_cases = [
    "I'm a 25 year old male, 80kg, 180cm, I go to the gym 3x a week and want to gain muscle",
    "female, 30 yo, 60 kg, 165cm, sedentary job, want to lose fat",
    "I'm a 22 y.o guy weighing 75kg and 175cm tall, I'm very active and want to bulk",
    "woman 45 years old 70 kg 160 cm walks daily want to maintain",
]

for i, text in enumerate(test_cases, 1):
    agent = UserProfileAgent()
    agent.extract_from_text(text)
    print(f"Test {i}: {text[:60]}...")
    print(f"  Extracted: {agent.profile}")
    missing = [k for k, v in agent.profile.items() if v is None]
    print(f"  Missing  : {missing}")
    print()

Test 1: I'm a 25 year old male, 80kg, 180cm, I go to the gym 3x a we...
  Extracted: {'age': 25, 'gender': 'male', 'weight_kg': 80.0, 'height_cm': 180.0, 'activity_level': 'moderate', 'goal': 'muscle_gain'}
  Missing  : []

Test 2: female, 30 yo, 60 kg, 165cm, sedentary job, want to lose fat...
  Extracted: {'age': 30, 'gender': 'male', 'weight_kg': 60.0, 'height_cm': 165.0, 'activity_level': 'sedentary', 'goal': 'weight_loss'}
  Missing  : []

Test 3: I'm a 22 y.o guy weighing 75kg and 175cm tall, I'm very acti...
  Extracted: {'age': 22, 'gender': 'male', 'weight_kg': 75.0, 'height_cm': 175.0, 'activity_level': 'active', 'goal': 'muscle_gain'}
  Missing  : []

Test 4: woman 45 years old 70 kg 160 cm walks daily want to maintain...
  Extracted: {'age': 45, 'gender': 'male', 'weight_kg': 70.0, 'height_cm': 160.0, 'activity_level': 'light', 'goal': 'maintenance'}
  Missing  : []



## Pipeline Output
What Agent 1 hands off to Agent 2

In [4]:
def agent1_output_format(profile: dict) -> dict:
    """Converts Agent 1 profile keys to Agent 2 expected format"""
    return {
        "age":            profile["age"],
        "sex":            profile["gender"],
        "height_cm":      profile["height_cm"],
        "weight_kg":      profile["weight_kg"],
        "activity_level": profile["activity_level"],
        "goal":           profile["goal"]
    }


# Simulate a completed profile (without interactive input)
sample_profile = {
    "age": 25,
    "gender": "male",
    "weight_kg": 80.0,
    "height_cm": 180.0,
    "activity_level": "moderate",
    "goal": "muscle_gain"
}

agent2_input = agent1_output_format(sample_profile)
print("Agent 1 → Agent 2 handoff:")
print(agent2_input)

Agent 1 → Agent 2 handoff:
{'age': 25, 'sex': 'male', 'height_cm': 180.0, 'weight_kg': 80.0, 'activity_level': 'moderate', 'goal': 'muscle_gain'}


## Run (interactive)
Uncomment to run the full interactive flow

In [5]:
# agent1 = UserProfileAgent()
# profile = agent1.run()
# agent2_input = agent1_output_format(profile)
# print("\nHandoff to Agent 2:")
# print(agent2_input)